# 6단계 권장 티어 후속: 여러 시점에 걸친 롤링 백테스트

**이 노트북은 Colab에서 직접 실행하는 걸 전제로 만들었다.** 런타임을 GPU(T4)로 바꾸고 시작할 것.

**왜 필요한가**: `06b_timeseries_dl_조.ipynb`(3차 시도)에서 화장품이 naive baseline 대비 7.0%, 생활이 20.6% 개선됐다고 나왔는데, **검증이 마지막 2개월, 딱 1번의 hold-out뿐이었다.** 시점을 하나만 봐서 우연히 좋게 나온 걸 수도 있다. 이번엔 각 상품마다 여러 시점에 걸쳐 반복적으로(2개월씩 밀어가면서) 예측→비교를 여러 번 해서, 그 개선이 시점에 상관없이 안정적인지 확인한다 (`darts`의 `historical_forecasts` 기능 사용).

**방법**: 각 상품 시계열의 앞 75%로만 모델을 1번 학습시키고(재학습 없음, `retrain=False`), 나머지 뒤 25% 구간을 2개월씩 밀어가며(stride=2) 여러 번 예측해서 실제값과 비교한다. 같은 시점들에 대해 naive(마지막 관측값 그대로) baseline도 같이 계산해서 비교한다.

**06b와 다른 점**: 상품마다 여러 번 검증해야 하므로 관측기간이 더 긴 상품만 쓴다(10개월 이상 → **16개월 이상**). 그만큼 대상 상품 수는 줄어든다(생활 296→245개, 화장품 248→212개).

**스케일링 관련 주의**: darts의 `Scaler` 클래스는 여러 시리즈를 한 리스트로 fit하면 리스트 순서에 의존하는데, `historical_forecasts`가 반환하는 구조(상품별로 여러 개의 윈도우)와 맞춰 쓰기가 까다로워서, 이번엔 상품별 최소/최대값을 직접 계산해서 수동으로 스케일링한다(로컬에서 자동 스케일러 대신 이 방식이 안전함을 검증함).

In [ ]:
!pip install -q "darts[torch]"
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from darts import TimeSeries
from darts.models import DLinearModel, NHiTSModel
from darts.utils.likelihood_models import QuantileRegression

fontpath = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(fontpath)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# 왼쪽 파일 탭에 reviews.parquet을 업로드한 뒤 실행
DATA_PATH = "reviews.parquet"
reviews = pd.read_parquet(DATA_PATH)
print(f"reviews {len(reviews)}행 로드 완료")

In [ ]:
MIN_REVIEWS = 30
INPUT_LEN = 6
OUTPUT_LEN = 2
TRAIN_FRAC = 0.75   # 앞 75%로 학습, 뒤 25%를 롤링 백테스트 구간으로
# 뒤 25% 구간에서 stride=2로 최소 2번은 검증하려면 전체가 웬만큼 길어야 함 -- 16개월로 설정
# (06b의 10개월보다 엄격해진 기준. 대상 상품 수는 줄지만 여러 시점 검증을 위해 필요)
MIN_MONTHS = 16

def build_product_series(domain: str):
    sub = reviews[(reviews["Domain"] == domain) & reviews["RDate_parsed"].notna()].copy()
    counts_by_product = sub.groupby("ProductName").size()
    qualifying = counts_by_product[counts_by_product >= MIN_REVIEWS].index
    sub = sub[sub["ProductName"].isin(qualifying)].copy()
    sub["month"] = sub["RDate_parsed"].dt.to_period("M").dt.to_timestamp()

    names, series_list, count_series_list = [], [], []
    skipped_short = 0
    for name, g in sub.groupby("ProductName"):
        m = g.groupby("month").agg(
            total=("review_id", "count"),
            negative=("GeneralPolarity", lambda s: (s == -1).sum()),
        )
        m["neg_ratio"] = m["negative"] / m["total"]
        full_idx = pd.date_range(m.index.min(), m.index.max(), freq="MS")
        if len(full_idx) < MIN_MONTHS:
            skipped_short += 1
            continue
        ratio = m["neg_ratio"].reindex(full_idx).interpolate(limit_direction="both")
        counts = m["total"].reindex(full_idx).fillna(0)

        names.append(name)
        series_list.append(TimeSeries.from_times_and_values(full_idx, ratio.values))
        count_series_list.append(TimeSeries.from_times_and_values(full_idx, counts.values))

    print(f"[{domain}] 리뷰 {MIN_REVIEWS}건 이상 상품 {len(qualifying)}개 중, "
          f"관측기간 {MIN_MONTHS}개월 이상인 {len(series_list)}개 사용 (짧아서 제외 {skipped_short}개)")
    return names, series_list, count_series_list

In [ ]:
def manual_scale(ts, lo, hi):
    return TimeSeries.from_times_and_values(ts.time_index, (ts.values() - lo) / (hi - lo + 1e-9))

def run_backtest(domain: str):
    names, series_list, cov_list = build_product_series(domain)

    # 상품별로 학습구간(앞 75%)의 최소/최대값만으로 스케일링 파라미터를 구한다 (검증구간 값은 안 봄 -- 미래 정보 누설 방지)
    train_mins, train_maxs, series_scaled, cov_scaled, train_target = [], [], [], [], []
    for s, c in zip(series_list, cov_list):
        cut = int(len(s) * TRAIN_FRAC)
        tmin, tmax = float(s[:cut].values().min()), float(s[:cut].values().max())
        cmin, cmax = float(c[:cut].values().min()), float(c[:cut].values().max())
        train_mins.append(tmin); train_maxs.append(tmax)
        series_scaled.append(manual_scale(s, tmin, tmax))
        cov_scaled.append(manual_scale(c, cmin, cmax))
        train_target.append(manual_scale(s, tmin, tmax)[:cut])

    # DLinear: covariate 없이 baseline으로 (06b와 동일한 역할)
    dlinear = DLinearModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                            n_epochs=50, random_state=42, pl_trainer_kwargs={"enable_progress_bar": False})
    dlinear.fit(series=train_target)
    dlinear_hist = dlinear.historical_forecasts(
        series=series_scaled, forecast_horizon=OUTPUT_LEN, stride=OUTPUT_LEN, start=TRAIN_FRAC,
        retrain=False, last_points_only=False, verbose=False,
    )

    # NHiTS + covariate (06b 3차 시도와 동일한 설계)
    nhits = NHiTSModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                        n_epochs=75, random_state=42,
                        likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
                        pl_trainer_kwargs={"enable_progress_bar": False})
    nhits.fit(series=train_target, past_covariates=cov_scaled)
    nhits_hist = nhits.historical_forecasts(
        series=series_scaled, past_covariates=cov_scaled,
        forecast_horizon=OUTPUT_LEN, stride=OUTPUT_LEN, start=TRAIN_FRAC,
        retrain=False, last_points_only=False, verbose=False, num_samples=200,
    )

    # 롤링 윈도우마다: DLinear, NHiTS, naive(직전 관측값) 셋 다 비교
    dlinear_errs, nhits_errs, naive_errs = [], [], []
    n_windows = 0
    for i, (dl_windows, nh_windows) in enumerate(zip(dlinear_hist, nhits_hist)):
        full_pd = series_list[i].to_series()
        for dl_w, nh_w in zip(dl_windows, nh_windows):
            n_windows += 1
            actual = full_pd.reindex(dl_w.time_index).values

            dl_pred = dl_w.values().ravel() * (train_maxs[i] - train_mins[i]) + train_mins[i]
            nh_pred = nh_w.quantile(0.5).values().ravel() * (train_maxs[i] - train_mins[i]) + train_mins[i]

            naive_origin = dl_w.time_index[0] - pd.DateOffset(months=1)
            naive_val = full_pd.get(naive_origin, np.nan)
            if pd.isna(naive_val):
                continue

            dlinear_errs.append(np.abs(actual - dl_pred).mean())
            nhits_errs.append(np.abs(actual - nh_pred).mean())
            naive_errs.append(np.abs(actual - naive_val).mean())

    print(f"[{domain}] 총 롤링 윈도우 {n_windows}개 (상품 {len(names)}개 x 상품마다 여러 시점)")
    print(f"[{domain}] naive 평균 MAE:   {np.mean(naive_errs):.4f}")
    print(f"[{domain}] DLinear 평균 MAE: {np.mean(dlinear_errs):.4f}")
    print(f"[{domain}] NHiTS 평균 MAE:   {np.mean(nhits_errs):.4f}")

    return {
        "n_windows": n_windows,
        "naive_mae": np.mean(naive_errs), "dlinear_mae": np.mean(dlinear_errs), "nhits_mae": np.mean(nhits_errs),
        "naive_errs": naive_errs, "dlinear_errs": dlinear_errs, "nhits_errs": nhits_errs,
    }

## 생활

In [ ]:
result_생활 = run_backtest("생활")

## 화장품

In [ ]:
result_화장품 = run_backtest("화장품")

## 통계적 유의성 검정 (부트스트랩·Wilcoxon, 손이 패션에서 쓴 것과 동일한 방법)

평균 MAE가 낮다고 바로 "유의미하다"고 할 수 없어서, 두 가지 독립적인 검정을 추가한다.
1. **부트스트랩 신뢰구간**: (naive오차-NHiTS오차) 윈도우별 차이를 10,000번 재표본해서 평균 차이의 95% CI가 0을 포함하는지 확인.
2. **Wilcoxon 부호순위검정**: 윈도우별 오차 쌍에 대해 비모수 검정.

두 검정이 다른 결론을 낼 수 있다 — naive가 오차 0으로 이기는 "조용한" 윈도우가 워낙 많으면(이 프로젝트 데이터의 특징), 평균은 NHiTS가 유리해도(부트스트랩 유의) 승패 기준으로는 naive가 더 자주 이길 수 있다(Wilcoxon 비유의). 두 결과를 다 보고 정확하게 해석할 것.

In [ ]:
from scipy import stats

def significance_test(domain: str, result: dict):
    naive_errs = np.array(result["naive_errs"])
    nhits_errs = np.array(result["nhits_errs"])
    diff = naive_errs - nhits_errs  # 양수면 NHiTS가 더 정확

    rng = np.random.default_rng(42)
    boot_means = np.array([rng.choice(diff, size=len(diff), replace=True).mean() for _ in range(10000)])
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

    stat, p_value = stats.wilcoxon(naive_errs, nhits_errs, alternative="greater")

    print(f"[{domain}] 윈도우 {len(diff)}개")
    print(f"  부트스트랩 95% CI: [{ci_low:.4f}, {ci_high:.4f}]",
          "-- 0 미포함(유의)" if ci_low > 0 else "-- 0 포함(비유의)")
    print(f"  Wilcoxon p-value: {p_value:.4f}", "-- 유의(p<0.05)" if p_value < 0.05 else "-- 비유의")
    return {"도메인": domain, "부트스트랩_CI_하한": ci_low, "부트스트랩_CI_상한": ci_high,
            "Wilcoxon_p값": p_value, "부트스트랩_유의": ci_low > 0, "Wilcoxon_유의": p_value < 0.05}

sig_생활 = significance_test("생활", result_생활)
sig_화장품 = significance_test("화장품", result_화장품)
sig_summary = pd.DataFrame([sig_생활, sig_화장품])
sig_summary.to_csv("06c_유의성검증_요약_조.csv", index=False, encoding="utf-8-sig")

from google.colab import files
files.download("06c_유의성검증_요약_조.csv")

## 결과 요약 + 다운로드

`06b`의 단일 hold-out 결과(생활 NHiTS 0.1182, 화장품 NHiTS 0.1801)와 이번 롤링 백테스트 결과를 비교해서, 개선폭이 시점에 상관없이 안정적으로 유지되는지 확인하면 된다.

In [ ]:
summary = pd.DataFrame([
    {"도메인": "생활", "롤링윈도우수": result_생활["n_windows"],
     "naive_MAE": result_생활["naive_mae"], "DLinear_MAE": result_생활["dlinear_mae"], "NHiTS_MAE": result_생활["nhits_mae"]},
    {"도메인": "화장품", "롤링윈도우수": result_화장품["n_windows"],
     "naive_MAE": result_화장품["naive_mae"], "DLinear_MAE": result_화장품["dlinear_mae"], "NHiTS_MAE": result_화장품["nhits_mae"]},
])
print(summary)
summary.to_csv("06c_롤링백테스트_요약_조.csv", index=False, encoding="utf-8-sig")

# 윈도우별 오차 분포도 같이 저장 (naive 대비 NHiTS가 이긴 윈도우 비율까지 확인 가능하게)
detail_생활 = pd.DataFrame({"naive": result_생활["naive_errs"], "dlinear": result_생활["dlinear_errs"], "nhits": result_생활["nhits_errs"]})
detail_화장품 = pd.DataFrame({"naive": result_화장품["naive_errs"], "dlinear": result_화장품["dlinear_errs"], "nhits": result_화장품["nhits_errs"]})
detail_생활.to_csv("06c_윈도우별오차_생활_조.csv", index=False, encoding="utf-8-sig")
detail_화장품.to_csv("06c_윈도우별오차_화장품_조.csv", index=False, encoding="utf-8-sig")

print()
print(f"생활: NHiTS가 naive보다 나은 윈도우 비율 = {(detail_생활['nhits'] < detail_생활['naive']).mean()*100:.1f}%")
print(f"화장품: NHiTS가 naive보다 나은 윈도우 비율 = {(detail_화장품['nhits'] < detail_화장품['naive']).mean()*100:.1f}%")

from google.colab import files
for fn in ["06c_롤링백테스트_요약_조.csv", "06c_윈도우별오차_생활_조.csv", "06c_윈도우별오차_화장품_조.csv"]:
    files.download(fn)